# 03 — Integrated Gradient Validation: 2D

Compare DDG integrated gradient against analytically integrated gradient
via the divergence theorem over dual cell polygons.

Covers: linear (machine precision), quadratic, cubic, trig fields.
Barycentric vs circumcentric comparison.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

from hyperct import Complex
from hyperct.ddg import compute_vd
from hyperct.ddg._dual_cell import dual_cell_polygon_2d, _shoelace_area
from hyperct._plotting import plot_complex

from ddgclib.operators.stress import scalar_gradient_integrated
from ddgclib.analytical import integrated_gradient_2d

In [ ]:
def make_mesh_2d(n_refine=1, method='barycentric', seed=None, jitter=0.05):
    HC = Complex(2, domain=[(0.0, 1.0), (0.0, 1.0)])
    HC.triangulate()
    for _ in range(n_refine): HC.refine_all()
    bV = set()
    for v in HC.V:
        on_bnd = any(abs(v.x_a[d]-lo) < 1e-14 or abs(v.x_a[d]-hi) < 1e-14
                     for d, (lo, hi) in enumerate([(0,1),(0,1)]))
        v.boundary = on_bnd
        if on_bnd: bV.add(v)
    if seed is not None:
        rng = np.random.default_rng(seed)
        for v in HC.V:
            if v not in bV and v.nn:
                el = min(np.linalg.norm(v.x_a - vn.x_a) for vn in v.nn)
                off = rng.uniform(-jitter*el, jitter*el, size=2)
                v.x = tuple(v.x_a[:2] + off)
                v.x_a = np.array(v.x, dtype=v.dtype)
    compute_vd(HC, method=method, cdist=1e-10)
    interior = [v for v in HC.V if v not in bV]
    return HC, bV, interior

def compute_errors(HC, interior, f_callable, include_mp=True):
    errors = {}
    for v in interior:
        num = scalar_gradient_integrated(v, HC, dim=2, field_attr='f')
        polygon = dual_cell_polygon_2d(v, include_edge_midpoints=include_mp)
        ana = integrated_gradient_2d(f_callable, polygon)
        errors[v] = np.linalg.norm(num - ana)
    return errors

## Linear field: f(x,y) = 3x - 2y + 1
Machine precision expected on all meshes with `barycentric_dual_p_ij`.

In [ ]:
f_lin = lambda x: 3*x[0] - 2*x[1] + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, seed, title in zip(axes, [None, 42], ['Symmetric', 'Jittered (seed=42)']):
    HC, bV, interior = make_mesh_2d(n_refine=1, seed=seed)
    for v in HC.V: v.f = f_lin(v.x_a[:2])
    errors = compute_errors(HC, interior, f_lin)

    # Plot mesh with gradient vectors
    plot_complex(HC, show=False, ax=ax)
    for v in interior:
        polygon = dual_cell_polygon_2d(v)
        patch = MplPolygon(polygon, closed=True, alpha=0.15, fc='C0', ec='C0')
        ax.add_patch(patch)
        # Numerical gradient arrow
        num = scalar_gradient_integrated(v, HC, dim=2, field_attr='f')
        scale = 0.3
        ax.arrow(v.x_a[0], v.x_a[1], num[0]*scale, num[1]*scale,
                 head_width=0.015, head_length=0.01, fc='red', ec='red')

    max_err = max(errors.values())
    ax.set_title(f'{title}\nmax error = {max_err:.2e}')
    ax.set_aspect('equal')

plt.suptitle('Linear f = 3x - 2y + 1: gradient arrows (red) on dual cells', y=1.02)
plt.tight_layout()
plt.show()

## Quadratic field: f(x,y) = x² + y²

In [ ]:
f_quad = lambda x: x[0]**2 + x[1]**2

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, seed, title in zip(axes, [None, 42], ['Symmetric', 'Jittered']):
    HC, bV, interior = make_mesh_2d(n_refine=2, seed=seed)
    for v in HC.V: v.f = f_quad(v.x_a[:2])
    errors = compute_errors(HC, interior, f_quad)

    plot_complex(HC, show=False, ax=ax)
    # Color dual cells by error
    err_vals = [errors[v] for v in interior]
    vmax = max(err_vals) if max(err_vals) > 0 else 1
    for v in interior:
        polygon = dual_cell_polygon_2d(v)
        c = plt.cm.Reds(errors[v] / vmax)
        patch = MplPolygon(polygon, closed=True, alpha=0.5, fc=c, ec='gray', lw=0.5)
        ax.add_patch(patch)

    ax.set_title(f'{title}\nmax error = {max(err_vals):.2e}')
    ax.set_aspect('equal')

plt.suptitle('Quadratic f = x² + y²: error heatmap on dual cells', y=1.02)
plt.tight_layout()
plt.show()

## Convergence: quadratic, cubic, trig fields

In [ ]:
import math

test_functions = [
    ('x² + y²', lambda x: x[0]**2 + x[1]**2),
    ('x³ + xy²', lambda x: x[0]**3 + x[0]*x[1]**2),
    ('sin(πx)cos(πy)', lambda x: math.sin(math.pi*x[0])*math.cos(math.pi*x[1])),
]

refines = [1, 2, 3, 4]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, seed, mesh_label in zip(axes, [None, 42], ['Symmetric', 'Jittered']):
    for fname, f in test_functions:
        max_errors = []
        for n_ref in refines:
            HC, bV, interior = make_mesh_2d(n_refine=n_ref, seed=seed)
            for v in HC.V: v.f = f(v.x_a[:2])
            errors = compute_errors(HC, interior, f)
            max_errors.append(max(errors.values()))
        ax.semilogy(refines, max_errors, 'o-', label=fname)

    ax.axhline(1e-14, color='g', ls='--', alpha=0.5, label='Machine ε')
    ax.set_xlabel('Refinement level')
    ax.set_ylabel('Max |error|')
    ax.set_title(f'Convergence ({mesh_label})')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Barycentric vs Circumcentric Comparison

In [ ]:
f_quad = lambda x: x[0]**2 + x[1]**2

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for col, method in enumerate(['barycentric', 'circumcentric']):
    for row, seed in enumerate([None, 42]):
        ax = axes[row, col]
        HC, bV, interior = make_mesh_2d(n_refine=2, method=method, seed=seed)
        for v in HC.V: v.f = f_quad(v.x_a[:2])
        errors = compute_errors(HC, interior, f_quad)

        plot_complex(HC, show=False, ax=ax)
        err_vals = [errors[v] for v in interior]
        vmax = max(err_vals) if max(err_vals) > 0 else 1
        for v in interior:
            polygon = dual_cell_polygon_2d(v)
            c = plt.cm.Reds(errors[v] / vmax)
            patch = MplPolygon(polygon, closed=True, alpha=0.5, fc=c, ec='gray', lw=0.5)
            ax.add_patch(patch)

        mesh_label = 'Symmetric' if seed is None else 'Jittered'
        ax.set_title(f'{method} / {mesh_label}\nmax err = {max(err_vals):.2e}')
        ax.set_aspect('equal')

plt.suptitle('f = x² + y²: Barycentric vs Circumcentric error comparison', y=1.02)
plt.tight_layout()
plt.show()